### Tensorflow 2.0

In [1]:
import tensorflow as tf

import keras

##### Customizing Models and Training Algorithms

In [3]:

#custom loss fuctions
def huber_fn(y_true,y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1 
    squared_loss = tf.square(error) / 2
    linear_loss = tf.abs(error) - 0.5 
    return tf.where(is_small_error , squared_loss ,linear_loss)

print("Función Huber loss personalizada definida.")

Función Huber loss personalizada definida.


### Custom Activation Fuctions , Initializers ,Regularizers , and Constraints

In [4]:
def my_softplus(z): # return value is just tf.nn.softplus(z)
    return tf.math.log(tf.exp(z) + 1.0)
def my_glorot_initializer(shape, dtype=tf.float32):
    stddev = tf.sqrt(2. / (shape[0] + shape[1]))
    return tf.random.normal(shape, stddev=stddev, dtype=dtype)
def my_l1_regularizer(weights):
    return tf.reduce_sum(tf.abs(0.01 * weights))
def my_positive_weights(weights): # return value is just tf.nn.relu(weights)
    return tf.where(weights < 0., tf.zeros_like(weights), weights)

In [5]:
layer = keras.layers.Dense(30, activation=my_softplus,
kernel_initializer=my_glorot_initializer,
kernel_regularizer=my_l1_regularizer,
kernel_constraint=my_positive_weights)

## 1. Tensores y Operaciones Básicas

Los tensores son la estructura fundamental en TensorFlow. Un tensor es un array multidimensional con tipo de dato uniforme.

In [6]:
# Crear tensores desde arrays
tensor1d = tf.constant([1, 2, 3, 4])  # Tensor 1D
tensor2d = tf.constant([[1, 2], [3, 4], [5, 6]])  # Tensor 2D
tensor3d = tf.constant([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])  # Tensor 3D

print("Tensor 1D:", tensor1d)
print("Tensor 2D shape:", tensor2d.shape)
print("Tensor 3D dtype:", tensor3d.dtype)

# Operaciones básicas
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])

suma = tf.add(a, b)  # Suma
resta = a - b  # Resta (operador)
multiplicacion = tf.multiply(a, b)  # Multiplicación elemento a elemento
producto_punto = tf.reduce_sum(a * b)  # Producto punto

print("\nSuma:", suma)
print("Producto punto:", producto_punto)

Tensor 1D: tf.Tensor([1 2 3 4], shape=(4,), dtype=int32)
Tensor 2D shape: (3, 2)
Tensor 3D dtype: <dtype: 'int32'>

Suma: tf.Tensor([5. 7. 9.], shape=(3,), dtype=float32)
Producto punto: tf.Tensor(32.0, shape=(), dtype=float32)


## 2. Variables en TensorFlow

Las variables son tensores mutables que se pueden entrenar. Son fundamentales para almacenar pesos y sesgos de modelos.

In [7]:
# Crear variables
variable = tf.Variable([1.0, 2.0, 3.0])  # Variable mutable
print("Variable inicial:", variable)

# Modificar variable
variable.assign([4.0, 5.0, 6.0])  # Asignar nuevo valor
print("Variable después de assign:", variable)

# Operaciones en-lugar (in-place)
variable.assign_add([1.0, 1.0, 1.0])  # Sumar
print("Después de assign_add:", variable)

# Variable trainable (por defecto)
w = tf.Variable(tf.random.normal((3, 2)), trainable=True)
b = tf.Variable(tf.zeros((2,)), trainable=False)  # No trainable

print("\nPeso w - trainable:", w.trainable)
print("Sesgo b - trainable:", b.trainable)

Variable inicial: <tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([1., 2., 3.], dtype=float32)>
Variable después de assign: <tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([4., 5., 6.], dtype=float32)>
Después de assign_add: <tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([5., 6., 7.], dtype=float32)>

Peso w - trainable: True
Sesgo b - trainable: False


## 3. Estructuras de Datos Especiales

TensorFlow proporciona estructuras especializadas para casos específicos como datos irregulares o dispersos.

In [8]:
# RaggedTensor: Tensor con filas de longitudes variables
ragged = tf.ragged.constant([[1, 2, 3], [4, 5], [6, 7, 8, 9]])
print("RaggedTensor:")
print(ragged)
print("Shape:", ragged.shape)

# SparseTensor: Tensor disperso (eficiente para datos con muchos ceros)
indices = [[0, 1], [1, 0], [2, 2]]  # Posiciones de valores no-cero
values = [10, 20, 30]  # Valores
shape = [3, 3]  # Forma del tensor

sparse = tf.SparseTensor(indices=indices, values=values, dense_shape=shape)
print("\nSparseTensor:")
print(sparse)

# Convertir a denso
dense = tf.sparse.to_dense(sparse)
print("Versión densa:")
print(dense)

RaggedTensor:
<tf.RaggedTensor [[1, 2, 3], [4, 5], [6, 7, 8, 9]]>
Shape: (3, None)

SparseTensor:
SparseTensor(indices=tf.Tensor(
[[0 1]
 [1 0]
 [2 2]], shape=(3, 2), dtype=int64), values=tf.Tensor([10 20 30], shape=(3,), dtype=int32), dense_shape=tf.Tensor([3 3], shape=(2,), dtype=int64))
Versión densa:
tf.Tensor(
[[ 0 10  0]
 [20  0  0]
 [ 0  0 30]], shape=(3, 3), dtype=int32)


## 4. TF Functions: Convirtiendo Código a Grafos

Las funciones de TensorFlow (@tf.function) convierten código Python en grafos computacionales optimizados para mayor rendimiento.

In [9]:
# Función sin @tf.function (Eager execution)
def eager_forward(x, w, b):
    return tf.matmul(x, w) + b

# Función con @tf.function (Grafo compilado)
@tf.function
def graph_forward(x, w, b):
    return tf.matmul(x, w) + b

# Ejemplos
x = tf.random.normal((100, 10))
w = tf.random.normal((10, 5))
b = tf.zeros((5,))

# Comparar rendimiento
import time

# Eager
start = time.time()
for _ in range(100):
    eager_forward(x, w, b)
eager_time = time.time() - start

# Graph
start = time.time()
for _ in range(100):
    graph_forward(x, w, b)
graph_time = time.time() - start

print(f"Tiempo eager: {eager_time:.4f}s")
print(f"Tiempo grafo: {graph_time:.4f}s")
print(f"Speedup: {eager_time/graph_time:.2f}x")

Tiempo eager: 0.0450s
Tiempo grafo: 0.0960s
Speedup: 0.47x


## 5. AutoGraph y Tracing

AutoGraph convierte automáticamente sentencias de control Python (if, while, for) en operaciones de grafo. El tracing determina el grafo para cada combinación única de tipos de entrada.

In [10]:
# AutoGraph convierte condicionales Python en ops de TF
@tf.function
def conditional_func(x):
    if x > 0:
        result = x * 2
    else:
        result = x / 2
    return result

print("Resultado positivo:", conditional_func(5.0).numpy())
print("Resultado negativo:", conditional_func(-4.0).numpy())

# Tracing: Cada tipo de entrada único genera un nuevo grafo
@tf.function
def traced_func(x):
    print(f"Trazando con tipo: {type(x)}")
    return x + 1

# Primera llamada con tensor - genera grafo
print("\nPrimera llamada:")
traced_func(tf.constant(1.0))

# Segunda llamada con mismo tipo - reutiliza grafo (sin tracing)
print("Segunda llamada (mismo tipo):")
traced_func(tf.constant(2.0))

# Tercera llamada con tipo diferente - nuevo grafo
print("Tercera llamada (tipo diferente):")
traced_func(tf.constant([1.0, 2.0]))

Resultado positivo: 10.0
Resultado negativo: -2.0

Primera llamada:
Trazando con tipo: <class 'tensorflow.python.framework.ops.SymbolicTensor'>
Segunda llamada (mismo tipo):
Tercera llamada (tipo diferente):
Trazando con tipo: <class 'tensorflow.python.framework.ops.SymbolicTensor'>


<tf.Tensor: shape=(2,), dtype=float32, numpy=array([2., 3.], dtype=float32)>

## 6. Modelo Completo Integrando Todo

Crearemos una red neuronal personalizada que utiliza:
- Variables de TensorFlow para pesos y sesgos
- Operaciones personalizadas
- Funciones de pérdida custom
- Regularización personalizada
- @tf.function para optimización
- AutoGraph en el entrenamiento

In [11]:
# 1. Crear datos sintéticos para entrenamiento
np = __import__('numpy')
X_train = tf.constant(np.random.randn(1000, 20).astype(np.float32))
y_train = tf.constant(np.random.randint(0, 2, (1000,)).astype(np.float32))

# 2. Red Neuronal Personalizada (usando bajo nivel API)
class CustomNeuralNetwork(tf.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # Variables: Pesos y sesgos
        self.w1 = tf.Variable(
            tf.random.normal((input_size, hidden_size)) * 0.01,
            trainable=True
        )
        self.b1 = tf.Variable(tf.zeros((hidden_size,)), trainable=True)
        
        self.w2 = tf.Variable(
            tf.random.normal((hidden_size, output_size)) * 0.01,
            trainable=True
        )
        self.b2 = tf.Variable(tf.zeros((output_size,)), trainable=True)
    
    # Forward pass con operaciones personalizadas
    @tf.function
    def __call__(self, x):
        # Capa 1: operación personalizada
        h = tf.matmul(x, self.w1) + self.b1
        # Activación custom (similar a softplus pero personalizado)
        h = tf.nn.relu(h)
        
        # Capa 2: 
        logits = tf.matmul(h, self.w2) + self.b2
        return logits

# 3. Función de pérdida personalizada (Huber Loss)
@tf.function
def huber_loss(y_true, y_pred):
    error = tf.abs(y_true - tf.squeeze(y_pred))
    is_small_error = error < 1.0
    squared_loss = tf.square(error) / 2
    linear_loss = error - 0.5
    return tf.reduce_mean(
        tf.where(is_small_error, squared_loss, linear_loss)
    )

# 4. Regularizador personalizado (L1 con factor)
@tf.function
def l1_regularization(variables, factor=0.01):
    return factor * tf.add_n([tf.reduce_sum(tf.abs(v)) for v in variables])

# 5. Inicializar modelo
model = CustomNeuralNetwork(input_size=20, hidden_size=32, output_size=1)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

print("Modelo creado con:")
print(f"  - Pesos capa 1: {model.w1.shape}")
print(f"  - Sesgos capa 1: {model.b1.shape}")
print(f"  - Pesos capa 2: {model.w2.shape}")
print(f"  - Sesgos capa 2: {model.b2.shape}")

Modelo creado con:
  - Pesos capa 1: (20, 32)
  - Sesgos capa 1: (32,)
  - Pesos capa 2: (32, 1)
  - Sesgos capa 2: (1,)


In [12]:
# 6. Paso de entrenamiento personalizado con @tf.function
# Esto demuestra cómo AutoGraph y TF Functions optimizan el entrenamiento

@tf.function
def train_step(model, optimizer, x, y):
    """
    Paso de entrenamiento compilado en grafo.
    Demuestra:
    - Uso de GradientTape (bajo nivel)
    - Cálculo de gradientes
    - Aplicación de optimizador
    - Integración con pérdida personalizada y regularización
    """
    with tf.GradientTape() as tape:
        # Forward pass
        predictions = model(x)
        
        # Pérdida custom
        loss = huber_loss(y, predictions)
        
        # Regularización personalizada
        reg_loss = l1_regularization(model.trainable_variables)
        
        # Pérdida total
        total_loss = loss + reg_loss
    
    # Gradientes
    gradients = tape.gradient(total_loss, model.trainable_variables)
    
    # Actualización
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return total_loss, loss, reg_loss

# 7. Entrenamiento
num_epochs = 5
batch_size = 32

print("Iniciando entrenamiento con modelo personalizado...\n")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    # Crear batches
    dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    dataset = dataset.batch(batch_size)
    
    for x_batch, y_batch in dataset:
        total_loss, loss, reg = train_step(model, optimizer, x_batch, y_batch)
        epoch_loss += total_loss
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\n✓ Entrenamiento completado")

Iniciando entrenamiento con modelo personalizado...

Epoch 1/5 - Loss: 0.2108
Epoch 2/5 - Loss: 0.1499
Epoch 3/5 - Loss: 0.1477
Epoch 4/5 - Loss: 0.1462
Epoch 5/5 - Loss: 0.1450

✓ Entrenamiento completado


In [13]:
# 8. Predicción e Inspección
print("=== RESUMEN DEL MODELO ENTRENADO ===\n")

# Predicciones
test_samples = X_train[:5]  # Primeros 5 samples
predictions = model(test_samples)

print("Predicciones de muestra:")
print(predictions.numpy())

# Inspeccionar pesos entrenados
print("\nEstadísticas de pesos entrenados:")
print(f"W1 - Min: {tf.reduce_min(model.w1):.4f}, Max: {tf.reduce_max(model.w1):.4f}, Mean: {tf.reduce_mean(model.w1):.4f}")
print(f"W2 - Min: {tf.reduce_min(model.w2):.4f}, Max: {tf.reduce_max(model.w2):.4f}, Mean: {tf.reduce_mean(model.w2):.4f}")

# Cuántos pesos entrenables
total_params = 0
for v in model.trainable_variables:
    params = tf.reduce_prod(v.shape).numpy()
    total_params += params
    print(f"Variable {v.name}: {v.shape} ({params} parámetros)")

print(f"\nTotal de parámetros entrenables: {total_params}")

print("\n=== CONCEPTOS DEMOSTRADOS ===")
print("✓ Tensores: Creados y manipulados en operaciones")
print("✓ Variables: Pesos y sesgos trainables")
print("✓ Operaciones: Matmul, add, funciones custom")
print("✓ Funciones Custom: Huber loss, regularización L1")
print("✓ TF Functions: @tf.function para optimización")
print("✓ AutoGraph: Integración automática en train_step")
print("✓ GradientTape: Cálculo automático de gradientes")
print("✓ Optimizadores: Adam actualizando variables")

=== RESUMEN DEL MODELO ENTRENADO ===

Predicciones de muestra:
[[0.47064757]
 [0.48985717]
 [0.4804503 ]
 [0.48515934]
 [0.4898493 ]]

Estadísticas de pesos entrenados:
W1 - Min: -0.0143, Max: 0.0164, Mean: -0.0000
W2 - Min: -0.0031, Max: 0.2146, Mean: 0.0067
Variable Variable:0: (32,) (32 parámetros)
Variable Variable:0: (1,) (1 parámetros)
Variable Variable:0: (20, 32) (640 parámetros)
Variable Variable:0: (32, 1) (32 parámetros)

Total de parámetros entrenables: 705

=== CONCEPTOS DEMOSTRADOS ===
✓ Tensores: Creados y manipulados en operaciones
✓ Variables: Pesos y sesgos trainables
✓ Operaciones: Matmul, add, funciones custom
✓ Funciones Custom: Huber loss, regularización L1
✓ TF Functions: @tf.function para optimización
✓ AutoGraph: Integración automática en train_step
✓ GradientTape: Cálculo automático de gradientes
✓ Optimizadores: Adam actualizando variables
